# 04 진단 모델 평가

03에서 만든 점수표를 학습에 안 쓴 평가 데이터(20%)로 확인한다.

- 근거: 데이터 처리 기획서 3-10 평가 (구간별 정답 비율, 구간별 사람 비율, 점수표와 모델 일치, AUC 기록만), 7-1 확인 시나리오 7, 8, 9
- 읽는 파일: 점수표 `ml/app/artifacts/credit_score_table.json`, 학습 기록 `ml/data/model/credit_train_log.json`, 정답과 나누기 `ml/data/model/credit_labels.csv` (03이 만듦)
- 점수 계산은 `ml/app/credit.py`의 `score_answers` (/credit/score와 같은 함수)
- 평가 결과는 학습 기록의 `eval`에 덧붙임 / GitHub에 올리기 전에 **출력 지우기**
- 맨 아래 **6. 평가 결과 보고서**

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ml 폴더 찾기 (노트북 위치: ml/data/ipynb)
ML_DIR = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "app" / "clean.py").exists())
sys.path.insert(0, str(ML_DIR))

from app.credit import OPTIONS, SOURCE_COLUMNS, load_score_table, score_answers, to_answers

CSV_DIR = ML_DIR / "data" / "csv"
TRAIN_LOG_PATH = ML_DIR / "data" / "model" / "credit_train_log.json"
LABELS_PATH = ML_DIR / "data" / "model" / "credit_labels.csv"

table = load_score_table()
log = json.loads(TRAIN_LOG_PATH.read_text(encoding="utf-8"))
labels = pd.read_csv(LABELS_PATH, index_col="row")
academy = pd.read_csv(CSV_DIR / "financial_credit_synthetic_155746x207.csv", usecols=SOURCE_COLUMNS)
assert table["version"] == log["version"], "점수표와 학습 기록 버전이 다름 (03을 다시 실행)"

test = labels["test"].values
answers = to_answers(academy)[test]
y = labels["y"].values[test]
NICE_SHARE = table["nice_share"]
print("버전:", table["version"], "/ 평가 데이터:", len(answers), "행 / 연체율 %.2f%%" % (y.mean() * 100))

## 1. 평가 데이터 점수 (`score_answers`)
상한 적용 점수와 상한 전 점수 (상한 전은 2번 모델 일치 확인에 씀)

In [ ]:
codes = {int(q[1:]): answers[q].cat.codes.values + 1 for q in OPTIONS}      # 보기 번호 (1부터)


def scores_of(score_table):
    result = []
    for i in range(len(answers)):
        row = {no: [int(c[i])] for no, c in codes.items()}
        row[5] = []
        result.append(score_answers(score_table, row)[0])
    return np.array(result)


score = scores_of(table)
score_no_cap = scores_of({**table, "caps": []})
print("점수 범위: %d ~ %d" % (score.min(), score.max()))

## 2. 점수표와 모델 일치 (기획서 3-10 평가)
학습 기록의 계수로 모델 로그 오즈를 계산해 같은 경계표로 바꾼 점수와, 점수표로 더한 점수(상한 전)가 같은지

In [ ]:
merge = log["merge"]
risk = np.full(len(answers), log["intercept"])
for q in log["score_questions"]:
    groups = answers[q].astype(str).map(lambda o: merge.get(q, {}).get(o, o))
    risk = risk + groups.map(lambda g: log["coef"].get(f"{q}_{g}", 0.0)).values
model_score = np.interp(log["offset"] - log["scale"] * risk, table["edges"]["linear"], table["edges"]["score"])
model_score = np.clip(model_score, 1, 1000)

diff = np.abs(score_no_cap - model_score)
print("점수표 - 모델 차이: 최대 %.2f점, 평균 %.3f점 (반올림만이면 1점 안쪽)" % (diff.max(), diff.mean()))

## 3. 구간별 사람 비율과 정답 비율 (기획서 3-10 평가)
- 정답 비율(연체율)이 900 초과 → 600 이하로 갈수록 계속 높아져야 통과
- 한 구간에 몰리지 않는지 (NICE 분포와 비교)

In [ ]:
def band_of(points):
    return np.select([points > 900, points > 800, points > 700, points > 600], ["BAND1", "BAND2", "BAND3", "BAND4"], "BAND5")


band = pd.Series(band_of(score))
band_before = pd.Series(band_of(score_no_cap))
rate = (pd.Series(y).groupby(band).mean() * 100).reindex(NICE_SHARE).round(2)
result = pd.DataFrame({"NICE%": NICE_SHARE,
                       "상한 전%": (band_before.value_counts(normalize=True) * 100).reindex(NICE_SHARE).round(1),
                       "상한 후%": (band.value_counts(normalize=True) * 100).reindex(NICE_SHARE).round(1),
                       "연체율%": rate})
increasing = bool((rate.diff().dropna() > 0).all())
print("연체율이 구간마다 높아짐:", "통과" if increasing else "실패")
print("가장 많은 구간 비율: %.1f%% (NICE 47.9%%)" % result["상한 후%"].max())
result

## 4. AUC (기록만)
심은 정답식을 되찾은 정도라 실제 신용 성능이 아님 (기획서 3-10 평가)

In [ ]:
auc_model = roc_auc_score(y, risk)              # 모델 로그 오즈
auc_score = roc_auc_score(y, -score)            # 최종 점수 (낮을수록 위험, 상한 포함)
print("AUC 모델 %.3f / 최종 점수 %.3f (03 학습 때 평가 %.3f)" % (auc_model, auc_score, log["auc_test"]))

## 5. 확인 시나리오 (기획서 7-1)
- 7: 1년 안 연체가 있으면 구간 상한 적용
- 8: 문항 하나 '잘 모르겠어요' → 가운데 보기로 계산 ('정확도 낮음' 표시는 Spring, CREDIT-01)
- 9: 문항 4, 5, 6 답만 바꾸면 점수 같음

In [ ]:
best = {int(q[1:]): [int(np.argmax(v["points"])) + 1] for q, v in table["questions"].items()}   # 문항마다 점수가 가장 높은 보기
best[5] = []
best_score, _ = score_answers(table, best)

overdue = {**best, 1: [OPTIONS["Q1"].index("1회") + 1]}
overdue_score, _ = score_answers(table, overdue)

unknown = {**best, 2: [0]}
_, unknown_items = score_answers(table, unknown)

others = {**best, 4: [4], 5: [1, 2, 3], 6: [6]}
others_score, _ = score_answers(table, others)

checks = pd.DataFrame([
    ("7 연체 1회 → 최대 800점", f"{best_score}점 → {overdue_score}점", overdue_score <= 800),
    ("8 문항 2 모름 → 가운데 점수", f"{unknown_items[2]}점 (점수표 {table['questions']['Q2']['unknown_points']})",
     unknown_items[2] == round(table["questions"]["Q2"]["unknown_points"])),
    ("9 문항 4, 5, 6만 바꿈 → 점수 같음", f"{best_score}점 → {others_score}점", others_score == best_score),
], columns=["시나리오", "결과", "통과"])
checks

In [ ]:
# 평가 결과를 학습 기록에 덧붙임 (git에 안 올림)
log["eval"] = {"rows": int(len(answers)), "label_rate": round(float(y.mean()), 4),
               "table_model_max_diff": round(float(diff.max()), 2), "rate_increasing": increasing,
               "band_share": result["상한 후%"].to_dict(), "band_rate": rate.to_dict(),
               "auc_model": round(auc_model, 4), "auc_score": round(auc_score, 4),
               "scenarios_passed": bool(checks["통과"].all())}
TRAIN_LOG_PATH.write_text(json.dumps(log, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("학습 기록에 평가 결과 저장:", TRAIN_LOG_PATH.name)

## 6. 평가 결과 보고서 (2026-11-13 실행, 점수표 버전 credit-20261113)

평가 데이터: 03에서 학습에 쓰지 않은 20% (31,150행, 다시 만든 정답 연체율 9.73%)

### 6-1. 기획서 3-10 평가 기준

| 기준 | 결과 | 판정 |
| --- | --- | --- |
| 구간별 정답 비율이 900 초과 → 600 이하로 갈수록 계속 높아짐 | 5.78% → 8.60% → 13.41% → 17.08% → 28.93% | 통과 |
| 구간별 사람 비율이 한 구간에 몰리지 않음 | 가장 많은 구간 BAND1 46.2% (NICE 47.9%). 다섯 구간 모두 NICE 분포와 2%p 안쪽 (BAND4만 3.1% vs 1.2%, 상한에 걸린 사람이 내려옴) | 통과 |
| 점수표 계산 점수와 모델 확률로 바꾼 점수가 같음 | 상한 전 기준 최대 0.54점, 평균 0.21점 차이 (보기 점수 소수 둘째 자리 반올림만) | 통과 |
| AUC | 모델 0.661 / 최종 점수(상한 포함) 0.659 | 기록만. 심은 정답식의 복원이라 실제 신용 성능이 아님 |

### 6-2. 구간별 결과 (평가 데이터)

| 구간 | NICE | 상한 전 | 상한 후 | 연체율 |
| --- | --- | --- | --- | --- |
| BAND1 900 초과 | 47.9% | 47.0% | 46.2% | 5.78% |
| BAND2 801~900 | 21.6% | 22.0% | 20.8% | 8.60% |
| BAND3 701~800 | 24.7% | 24.9% | 25.3% | 13.41% |
| BAND4 601~700 | 1.2% | 1.4% | 3.1% | 17.08% |
| BAND5 600 이하 | 4.6% | 4.7% | 4.7% | 28.93% |

- 학습 데이터 결과(03 보고서 7-6)와 비율, 연체율이 거의 같음 → 학습 데이터에만 맞춘 결과가 아님
- 상한을 넣어도 AUC는 0.661 → 0.659로 거의 그대로

### 6-3. 확인 시나리오 (기획서 7-1)

| 번호 | 상황 | 결과 | 판정 |
| --- | --- | --- | --- |
| 7 | 1년 안 연체 1회 (나머지는 가장 좋은 답) | 1000점 → 800점 (상한) | 통과 |
| 8 | 문항 2 '잘 모르겠어요' | 문항 점수 −73점 = 점수표 가운데 점수 (보기 4개라 가운데 두 보기 평균) | 통과 ('정확도 낮음' 표시는 Spring, CREDIT-01) |
| 9 | 문항 4, 5, 6 답만 바꿈 | 1000점 → 1000점 | 통과 |

### 6-4. 남은 것과 한계

- 다음: FastAPI `/credit/score`에서 같은 `score_answers`와 점수표 파일로 계산 (시간 제한 3초, 답과 점수 로그 금지, B8, B14)
- 한계는 03 보고서 7-7과 같음 (교육용 합성 데이터, BAND4 사람 수가 적어 상한 경계가 흔들릴 수 있음)